In [1]:
# Customer Lifetime Value (LTV) Prediction
# Project: Customer Churn Prediction & LTV Engine
# Owner: Varsha

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
#Project path
import sys
from pathlib import Path

project_root = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

Project root: c:\Users\pranj\OneDrive\Desktop\customer-churn-ltv-engine


In [3]:
#Load LTV datasets
processed_dir = project_root / "data" / "processed"

X_train = pd.read_csv(processed_dir / "X_train.csv")
X_test = pd.read_csv(processed_dir / "X_test.csv")

y_ltv_train = pd.read_csv(
    processed_dir / "y_ltv_train.csv"
).squeeze()

y_ltv_test = pd.read_csv(
    processed_dir / "y_ltv_test.csv"
).squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_ltv_train:", y_ltv_train.shape)
print("y_ltv_test:", y_ltv_test.shape)

X_train: (5634, 92)
X_test: (1409, 92)
y_ltv_train: (5634,)
y_ltv_test: (1409,)


In [4]:
print("Missing values in X_train:")
print(X_train.isnull().sum().sum())

print("\nMissing values in X_test:")
print(X_test.isnull().sum().sum())

print("\nLTV training statistics:")
print(y_ltv_train.describe())

Missing values in X_train:
0

Missing values in X_test:
0

LTV training statistics:
count    5634.000000
mean     2299.334682
std      2279.204278
min         0.000000
25%       402.975000
50%      1394.925000
75%      3835.825000
max      8684.800000
Name: TotalCharges, dtype: float64


In [5]:
ltv_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

ltv_model.fit(X_train, y_ltv_train)

print("LTV model training completed successfully.")

LTV model training completed successfully.


In [6]:
RandomForestRegressor

sklearn.ensemble._forest.RandomForestRegressor

In [7]:
y_ltv_pred = ltv_model.predict(X_test)

print("Sample actual LTV:", y_ltv_test.head().tolist())
print("Sample predicted LTV:", y_ltv_pred[:5])

Sample actual LTV: [8468.2, 908.55, 3211.2, 1468.75, 5919.35]
Sample predicted LTV: [8479.8415   908.52275 3210.46275 1469.5425  5918.7095 ]


In [8]:
mae = mean_absolute_error(y_ltv_test, y_ltv_pred)

rmse = np.sqrt(
    mean_squared_error(y_ltv_test, y_ltv_pred)
)

r2 = r2_score(
    y_ltv_test,
    y_ltv_pred
)

print("LTV Model Performance")
print("---------------------")
print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

LTV Model Performance
---------------------
MAE  : 1.06
RMSE : 1.91
R²   : 1.0000


In [9]:
comparison = pd.DataFrame({
    "Actual_LTV": y_ltv_test.values,
    "Predicted_LTV": y_ltv_pred
})

comparison["Absolute_Error"] = (
    comparison["Actual_LTV"] -
    comparison["Predicted_LTV"]
).abs()

display(comparison.head(10))

,Actual_LTV,Predicted_LTV,Absolute_Error
0,8468.20,8479.84150,11.64150
1,908.55,908.52275,0.02725
2,3211.20,3210.46275,0.73725
3,1468.75,1469.54250,0.79250
4,5919.35,5918.70950,0.64050
5,1992.55,1992.86175,0.31175
6,1956.40,1955.43000,0.97000
7,439.20,438.93200,0.26800
8,1311.60,1313.22350,1.62350
9,770.60,770.13500,0.46500


In [10]:
import joblib

model_dir = project_root / "models" / "ltv"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "ltv_model.pkl"

joblib.dump(ltv_model, model_path)

print(f"LTV model saved successfully: {model_path}")

LTV model saved successfully: c:\Users\pranj\OneDrive\Desktop\customer-churn-ltv-engine\models\ltv\ltv_model.pkl


In [11]:
loaded_ltv_model = joblib.load(model_path)

sample_predictions = loaded_ltv_model.predict(
    X_test.head(5)
)

print("Sample predictions:", sample_predictions)
print("LTV model loading successful.")

Sample predictions: [8479.8415   908.52275 3210.46275 1469.5425  5918.7095 ]
LTV model loading successful.
